# Extension 2: prompted LLMs

This notebook asks large language models to spoil clickbait **without any training**: they only get a prompt. It saves every answer and its scores, for the comparison with our fine-tuned models in `2-compare.ipynb`.

| Model | Runs on | Settings |
|---|---|---|
| Llama 3.1 8B Instruct | this Mac, Ollama (4-bit) | greedy decoding. The same base model we fine-tuned in Milestone 3, so it shows what fine-tuning adds. |
| GPT-OSS 20B | this Mac, Ollama | low reasoning effort, greedy decoding |
| Claude Sonnet 5.5, Claude Haiku 5.5 | Anthropic API (direct requests; `BATCH=1`: Message Batches at half price) | low effort |
| GPT-6.1 Sol, GPT-6 Luna (ChatGPT) | OpenAI Responses API (direct requests; `BATCH=1`: Batch API at half price) | low reasoning effort |

The cloud models come in price-matched pairs: Sonnet 5.5 and GPT-6.1 Sol cost $2 / $10 per million input / output tokens, Haiku 5.5 and GPT-6 Luna $0.10 / $0.50.

Every model gets the same prompt (`spoilerqa/llm.py`):
- a system prompt that explains the three spoiler types and asks for the spoiler **copied word for word from the article**;
- *k* worked examples from the training set;
- the post and its article.

It answers in JSON, `{"type": ..., "spoilers": [...]}`, which each provider's structured-output option enforces.

**What the notebook runs:**
1. **Few-shot sweep on 200 dev posts.** Each model with 0, 3 and 6 worked examples. The best *k* per model, by dev BLEU, is used from then on. This is the parameter sweep for this extension.
2. **Main run on the full test set (1,000 posts).** Each model with its best *k*, reading the post and the article.
3. **Two probes on a fixed sample of 300 test posts:**
   - *zero-shot* (no worked examples);
   - *closed-book* (the post only, no article). If a model still finds the spoiler, it may know this public dataset or the news stories from its training data.

**Snapping.** Every predicted spoiler is replaced by the closest span of the article: an exact match ignoring case and spacing, or, failing that, a fuzzy match of at least 80/100. Answers that match nothing are kept as written. Scores are reported both with and without snapping. Closed-book answers are never snapped.

**How to run it** (on the Mac, from `milestone4/code`):
1. Put the API keys in a `.env` file in `milestone4/code/` or in the project root (it is git-ignored):
   ```
   ANTHROPIC_API_KEY=...
   OPENAI_API_KEY=...
   ```
   Models whose key is missing are skipped.
2. Make sure Ollama is running (`ollama list` shows `llama3.1:8b` and `gpt-oss:20b`).
3. Run it in a terminal, with `caffeinate` so the Mac doesn't sleep:
   ```bash
   caffeinate -i jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 1-prompted-llms.ipynb
   ```
   - It takes about 4–5 hours, mostly the two local models (about 3.5 s per post on the M4 Pro). The cloud models are asked directly, several requests at a time, and take minutes per run.
   - The measured time and cost of every test run are in `output/llm-test-runs.md`.
   - Every answer is saved as it arrives (`output/raw/`), so re-running continues where it stopped. If a post has no answer (a request that failed even after retries), the notebook stops with an error; re-running asks it again.
   - **Batch option:** `BATCH=1 caffeinate -i jupyter nbconvert ...` sends the cloud models through the Claude and OpenAI batch APIs instead, at half price. Batches can wait in the provider's queue for hours, and OpenAI limits how many tokens can be queued per model: 1.35M for GPT-6.1 Sol on our account, less than one 1,000-post run with 6 worked examples, so that batch failed. Our run started with batches and switched to direct requests for what was left. A batch still pending from an earlier run is collected first when running without `BATCH=1` (cancelled if unfinished; the answers it already has are kept).

**Smoke test:** `SMOKE=1 jupyter nbconvert --to notebook --execute 1-prompted-llms.ipynb --output smoke-run.ipynb`. It uses only the cheapest model of each backend (Llama 3.1 8B, Claude Haiku 5.5, GPT-6 Luna; the cloud ones called directly instead of in a batch, for a few cents) and 6 posts per step. Everything goes to `smoke/`.

## Setup

Imports, paths, the API keys from `.env`, the data and the post samples.
- The dev and test samples are stratified by spoiler type and drawn with a fixed seed, so every model sees exactly the same posts.

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from spoilerqa import llm
from spoilerqa.data import DEFAULT_DATA_DIR, TYPES, load_jsonl, load_split, post_text, spoiler_type, write_jsonl
from spoilerqa.metrics import as_text, bleu_scores, ensure_nltk_data, type_scores
from spoilerqa.tables import to_markdown

# API keys (ANTHROPIC_API_KEY, OPENAI_API_KEY) from a git-ignored .env here in code/ or in the project root
load_dotenv(Path(".env"))
load_dotenv(Path.cwd().resolve().parents[1] / ".env")
ensure_nltk_data()
pd.set_option("display.max_colwidth", 90)

SMOKE = os.environ.get("SMOKE", "0") == "1"
DATA_DIR = Path(os.environ.get("DATA_DIR", DEFAULT_DATA_DIR))
OUTPUT_DIR = Path("smoke/output") if SMOKE else Path("../output")
RAW_DIR, PRED_DIR, SCORE_DIR, GOLD_DIR = (OUTPUT_DIR / d for d in ("raw", "predictions", "scores", "gold"))
for d in (RAW_DIR, PRED_DIR, SCORE_DIR, GOLD_DIR):
    d.mkdir(parents=True, exist_ok=True)

train, dev, test = (load_split(DATA_DIR, s) for s in ("train", "dev", "test"))
DEV_N, PROBE_N = (6, 6) if SMOKE else (200, 300)
dev_sample = llm.stratified_sample(dev, DEV_N, seed=0)
test_probe = llm.stratified_sample(test, PROBE_N, seed=0)
test_main = llm.stratified_sample(test, 6, seed=1) if SMOKE else test
K_VALUES = [0, 3] if SMOKE else [0, 3, 6]
examples = {k: llm.fewshot_examples(train, k) for k in K_VALUES}

availability = {key: llm.backend_available(key) for key in llm.MODELS}
SMOKE_MODELS = {"llama3.1-8b", "claude-haiku-5.5", "gpt-6-luna"}  # the cheapest model of each backend
MODELS = [key for key, (ok, _) in availability.items() if ok and (not SMOKE or key in SMOKE_MODELS)]
BATCH = os.environ.get("BATCH", "0") == "1" and not SMOKE  # cloud models: direct requests unless BATCH=1
display(pd.DataFrame([{"model": key, "label": llm.MODELS[key]["label"], "available": ok, "note": note}
                      for key, (ok, note) in availability.items()]))
print("models in this run:", MODELS)
print(f"dev sample: {len(dev_sample)} posts, test probe sample: {len(test_probe)}, main test: {len(test_main)}")

/opt/anaconda3/envs/NLP/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,model,label,available,note
0,llama3.1-8b,"Llama 3.1 8B Instruct (Ollama, 4-bit)",True,ok
1,gpt-oss-20b,"GPT-OSS 20B (Ollama, low reasoning)",True,ok
2,claude-sonnet-5.5,Claude Sonnet 5.5 (low effort),True,ok
3,claude-haiku-5.5,Claude Haiku 5.5 (low effort),True,ok
4,gpt-6.1-sol,GPT-6.1 Sol (low reasoning effort),True,ok
5,gpt-6-luna,GPT-6 Luna (low reasoning effort),True,ok


models in this run: ['llama3.1-8b', 'gpt-oss-20b', 'claude-sonnet-5.5', 'claude-haiku-5.5', 'gpt-6.1-sol', 'gpt-6-luna']
dev sample: 200 posts, test probe sample: 300, main test: 1000


## The prompt

The system prompt and one complete 3-shot prompt, as every model receives it (the article is shortened here for display).

In [2]:
print(llm.SYSTEM_ARTICLE)
print("\n" + "=" * 100 + "\n")
system, messages = llm.build_prompt(dev_sample[0], "article", llm.fewshot_examples(train, 3))
for m in messages:
    text = m["content"] if len(m["content"]) < 600 else m["content"][:600] + " [...]"
    print(f"--- {m['role']}\n{text}\n")
print("--- gold answer for the last post:", llm.answer_json(dev_sample[0]))
print("\nclosed-book system prompt:\n" + llm.SYSTEM_CLOSED_BOOK)

You spoil clickbait. You get a clickbait post and the article it links to. The post teases something without saying it; the spoiler is the text from the article that gives it away.

Spoiler types:
- "phrase": one short answer of a few words, such as a name, a number or a thing.
- "passage": an answer that needs a sentence or two.
- "multi": several separate answers, for example when the post promises a list ("5 ways to ...") or asks about more than one thing. Give each part as its own spoiler, at most 5.

Copy each spoiler word for word from the article, using the shortest text that fully answers the post. Do not paraphrase, explain or add anything.

Reply with JSON only: {"type": "phrase" | "passage" | "multi", "spoilers": ["..."]}. For phrase and passage, give exactly one spoiler.


--- user
Post: Get a Pumpkin Spice Latte EARLY with this secret code

Article:
Get A Pumpkin Spice Latte Early With This Secret Code
Starbucks' cult classic, the Pumpkin Spice Latte, won't officially make

## Running models, making predictions, scoring

- **`run_jobs`** runs a list of (model, posts, setup, k, tag) jobs: the local models one after the other, then the cloud models (Claude and ChatGPT). With `BATCH=1` it submits all cloud batches first, so they run on the providers' side while the Mac works on the local models, and collects them at the end.
- **`make_predictions`** turns the raw answers into SemEval predictions, with or without snapping. If a post has no answer at all (a failed request or batch), it stops with an error instead of scoring an empty prediction; re-running the notebook asks the missing posts again.
- **`quick_scores`** gives BLEU overall and per type, plus the type balanced accuracy, for the dev sweep.
- **`run_score`** runs `score.py` (all official metrics) for the test runs.

In [3]:
def run_jobs(jobs):
    cloud = [j for j in jobs if llm.MODELS[j[0]]["backend"] in llm.BATCH_BACKENDS]
    local = sorted((j for j in jobs if j not in cloud), key=lambda j: j[0])  # one local model after the other
    if BATCH:  # submit every batch now; they run on the providers' side while the local models run here
        for model, records, setup, k, tag in cloud:
            llm.run_llm(model, records, setup, k, examples[k], RAW_DIR, tag, batch=True, wait=False)
    for model, records, setup, k, tag in local:
        llm.run_llm(model, records, setup, k, examples[k], RAW_DIR, tag)
    for model, records, setup, k, tag in cloud:
        llm.run_llm(model, records, setup, k, examples[k], RAW_DIR, tag, batch=BATCH, wait=True)


def load_raw(model, setup, k, tag):
    path = RAW_DIR / f"{llm.run_name(model, setup, k, tag)}.jsonl"
    return {row["uuid"]: row for row in load_jsonl(path)} if path.exists() else {}


def make_predictions(model, records, setup, k, tag, snap):
    raw = load_raw(model, setup, k, tag)
    missing = [r["uuid"] for r in records if r["uuid"] not in raw]
    if missing:
        raise RuntimeError(f"{llm.run_name(model, setup, k, tag)}: {len(missing)} of {len(records)} posts have no "
                           "answer (failed requests or a failed batch). Re-run the notebook to ask them again.")
    preds, infos = [], []
    for r in records:
        pred, info = llm.to_llm_prediction(r, raw[r["uuid"]], snap=snap)
        preds.append(pred)
        infos.append(info)
    return preds, infos


def quick_scores(records, preds):
    gold_text = [as_text(r["spoiler"]) for r in records]
    bleu = bleu_scores(gold_text, [as_text(p["spoiler"]) for p in preds])
    out = {"BLEU": sum(bleu) / len(bleu)}
    for t in TYPES:
        idx = [i for i, r in enumerate(records) if spoiler_type(r) == t]
        out[t] = sum(bleu[i] for i in idx) / len(idx) if idx else None
    out["type bal. acc."] = type_scores([spoiler_type(r) for r in records],
                                        [p["spoilerType"] for p in preds])["balanced-accuracy"]
    return out


def gold_file(records, tag):
    if len(records) == len(test) and not SMOKE:
        return DATA_DIR / "test.jsonl"
    path = GOLD_DIR / f"{tag}.jsonl"
    write_jsonl(records, path)
    return path


def run_score(name, records, tag):
    cmd = [sys.executable, "score.py", "--gold", str(gold_file(records, tag)), "--pred", str(PRED_DIR / f"{name}.jsonl"),
           "--output", str(SCORE_DIR / f"{name}.json")]
    if SMOKE:
        cmd.append("--no-bertscore")
    subprocess.run(cmd, capture_output=True, text=True, check=True)
    return json.loads((SCORE_DIR / f"{name}.json").read_text())


def save_and_score(model, records, setup, k, tag, snap_options=(True, False)):
    # Writes the prediction files (snapped and, for comparison, unsnapped) and scores them with score.py.
    name = llm.run_name(model, setup, k, tag)
    rows = {}
    for snap in snap_options:
        preds, infos = make_predictions(model, records, setup, k, tag, snap)
        suffix = "" if snap or setup == "closedbook" else "-nosnap"
        write_jsonl(preds, PRED_DIR / f"{name}{suffix}.jsonl")
        rows[snap] = (run_score(name + suffix, records, tag), infos)
    return rows

## 1. Few-shot sweep on the dev sample

Every model answers the 200 dev posts with 0, 3 and 6 worked examples. The table shows BLEU (with and without snapping) and type balanced accuracy. The best *k* per model (snapped dev BLEU; the smaller *k* on a tie) is used for the test runs.

In [4]:
DEV_TAG = f"dev{DEV_N}"
run_jobs([(m, dev_sample, "article", k, DEV_TAG) for m in MODELS for k in K_VALUES])

sweep_rows = []
for m in MODELS:
    for k in K_VALUES:
        for snap in (True, False):
            preds, infos = make_predictions(m, dev_sample, "article", k, DEV_TAG, snap)
            sweep_rows.append({"model": m, "k": k, "snapped": snap, **quick_scores(dev_sample, preds),
                               "unparsed": sum(not i["parsed"] for i in infos)})
sweep = pd.DataFrame(sweep_rows)
snapped = sweep[sweep["snapped"]].sort_values(["model", "BLEU", "k"], ascending=[True, False, True])
best_k = {m: int(snapped[snapped["model"] == m].iloc[0]["k"]) for m in MODELS}
sweep.to_csv(OUTPUT_DIR / "llm-dev-sweep.csv", index=False)
(OUTPUT_DIR / "llm-dev-sweep.md").write_text(to_markdown(sweep))
(OUTPUT_DIR / "llm-best-k.json").write_text(json.dumps(best_k, indent=2))
print("best k per model:", best_k)
sweep.pivot_table(index="model", columns=["snapped", "k"], values="BLEU").style.format(precision=4)

gpt-oss-20b_article_k0_dev200: 200 done, 0 to ask
gpt-oss-20b_article_k3_dev200: 200 done, 0 to ask
gpt-oss-20b_article_k6_dev200: 200 done, 0 to ask
llama3.1-8b_article_k0_dev200: 200 done, 0 to ask
llama3.1-8b_article_k3_dev200: 200 done, 0 to ask
llama3.1-8b_article_k6_dev200: 200 done, 0 to ask
claude-sonnet-5.5_article_k0_dev200: 200 done, 0 to ask
claude-sonnet-5.5_article_k3_dev200: 200 done, 0 to ask
claude-sonnet-5.5_article_k6_dev200: 200 done, 0 to ask
claude-haiku-5.5_article_k0_dev200: 200 done, 0 to ask
claude-haiku-5.5_article_k3_dev200: 200 done, 0 to ask
claude-haiku-5.5_article_k6_dev200: 200 done, 0 to ask
gpt-6.1-sol_article_k0_dev200: 200 done, 0 to ask
gpt-6.1-sol_article_k3_dev200: 200 done, 0 to ask
gpt-6.1-sol_article_k6_dev200: 200 done, 0 to ask
gpt-6-luna_article_k0_dev200: 200 done, 0 to ask
gpt-6-luna_article_k3_dev200: 200 done, 0 to ask
gpt-6-luna_article_k6_dev200: 200 done, 0 to ask


best k per model: {'llama3.1-8b': 0, 'gpt-oss-20b': 0, 'claude-sonnet-5.5': 6, 'claude-haiku-5.5': 3, 'gpt-6.1-sol': 6, 'gpt-6-luna': 3}


## 2. Main run: the full test set

Every model with its best *k*, reading the post and the article. Both prediction files (snapped, and `-nosnap`) are scored with `score.py`. The snapped one is the system's official output.

In [5]:
TEST_TAG = "smoke-main" if SMOKE else "test"
run_jobs([(m, test_main, "article", best_k[m], TEST_TAG) for m in MODELS])
main_scores = {m: save_and_score(m, test_main, "article", best_k[m], TEST_TAG) for m in MODELS}

gpt-oss-20b_article_k0_test: 1000 done, 0 to ask
llama3.1-8b_article_k0_test: 1000 done, 0 to ask
claude-sonnet-5.5_article_k6_test: 1000 done, 0 to ask
claude-haiku-5.5_article_k3_test: 1000 done, 0 to ask
gpt-6.1-sol_article_k6_test: 0 done, 1000 to ask


  25/1000 posts, 0.4 s/post, about 7 min left


  50/1000 posts, 0.3 s/post, about 5 min left


  75/1000 posts, 0.3 s/post, about 5 min left


  100/1000 posts, 0.3 s/post, about 4 min left


  125/1000 posts, 0.3 s/post, about 4 min left


... (32 outputs left out)

  950/1000 posts, 0.3 s/post, about 0 min left


  975/1000 posts, 0.3 s/post, about 0 min left


  1000/1000 posts, 0.3 s/post, about 0 min left


gpt-6-luna_article_k3_test: 1000 done, 0 to ask


## 3. Probes on 300 test posts: zero-shot and closed-book

- **Zero-shot:** the article but no worked examples (snapped, like the main run).
- **Closed-book:** the post only, with the model's best *k* worked examples (also without articles), and no snapping. A model that scores well here knows the answer without reading the article.

The same 300 posts are scored from the main run too, so all three setups are compared on identical posts in `2-compare.ipynb`.

In [6]:
PROBE_TAG = f"test{PROBE_N}"
run_jobs([(m, test_probe, "article", 0, PROBE_TAG) for m in MODELS]
         + [(m, test_probe, "closedbook", best_k[m], PROBE_TAG) for m in MODELS])
probe_scores = {}
for m in MODELS:
    probe_scores[(m, "zero-shot")] = save_and_score(m, test_probe, "article", 0, PROBE_TAG, snap_options=(True,))
    probe_scores[(m, "closed-book")] = save_and_score(m, test_probe, "closedbook", best_k[m], PROBE_TAG,
                                                      snap_options=(False,))

gpt-oss-20b_article_k0_test300: 300 done, 0 to ask
gpt-oss-20b_closedbook_k0_test300: 300 done, 0 to ask
llama3.1-8b_article_k0_test300: 300 done, 0 to ask
llama3.1-8b_closedbook_k0_test300: 300 done, 0 to ask
claude-sonnet-5.5_article_k0_test300: 300 done, 0 to ask
claude-haiku-5.5_article_k0_test300: 300 done, 0 to ask
gpt-6.1-sol_article_k0_test300: 300 done, 0 to ask
gpt-6-luna_article_k0_test300: 0 done, 300 to ask


  cancelled the unfinished batch batch_6af19a5f44448190bc68a549e5ad1c85; collecting the answers it already has
  resuming batch batch_6af19a5f44448190bc68a549e5ad1c85
  batch cancelling: 236/300 done


  batch cancelling: 236/300 done


  batch cancelling: 236/300 done


  batch cancelling: 236/300 done


  batch cancelling: 236/300 done


... (125 outputs left out)

  batch cancelling: 277/300 done


  batch cancelling: 278/300 done


  batch cancelled after 611 min: 278 answers, 22 failed (re-run to retry)
  first error: {'status_code': 503, 'request_id': '4091557e-8e27-486e-81db-ab60d478a578', 'body': {'error': {'message': 'Our servers are currently overloaded. Please try again later.', 'type': 'service_unavailable_error', 'param': None, 'code': 'server_is_overloaded'}}}


  22/22 posts, 0.6 s/post, about 0 min left


## 4. Summary of all test runs

One row per run:
- **scores:** BLEU overall and per type, METEOR, BERTScore and type balanced accuracy;
- **snapping:** the share of spoilers matched exactly, matched fuzzily, or kept as written;
- **unparsed:** answers that couldn't be read;
- **minutes:** wall time of the run (for a batch, this includes the time it waited in the provider's queue);
- **cost $:** at list prices, counted per answer: half price for answers that came from a batch, standard price for direct requests. The local models are free.

Written to `output/llm-test-runs.csv` and `.md`.

In [7]:
def run_minutes(name):
    log = RAW_DIR / "run-log.jsonl"
    entries = [e for e in load_jsonl(log) if e["run"] == name] if log.exists() else []
    return sum(e.get("minutes", 0) for e in entries) or None


def summary_row(m, setup_label, setup, k, tag, scores_by_snap):
    name = llm.run_name(m, setup, k, tag)
    snap = True if True in scores_by_snap else False
    s, infos = scores_by_snap[snap]
    hows = [h for i in infos for h in i["snap"]]
    raw_rows = list(load_raw(m, setup, k, tag).values())
    sp = s["spoiler"]
    row = {"model": m, "setup": setup_label, "k": k, "posts": s["n"], "BLEU": sp["all"]["bleu"],
           "BLEU unsnapped": scores_by_snap[False][0]["spoiler"]["all"]["bleu"] if False in scores_by_snap and snap else None,
           **{t: sp[t]["bleu"] for t in TYPES}, "METEOR": sp["all"].get("meteor"), "BERTScore": sp["all"].get("bertscore"),
           "type bal. acc.": s.get("type", {}).get("balanced-accuracy"),
           "exact / fuzzy / kept": " / ".join(f"{hows.count(h) / max(len(hows), 1):.0%}" for h in ("exact", "fuzzy", "kept")) if hows else "",
           "unparsed": sum(not i["parsed"] for i in infos),
           "minutes": run_minutes(name),
           "cost $": llm.run_cost(raw_rows, llm.MODELS[m]["model"])}  # batch or standard price per answer
    return row


summary = [summary_row(m, "few-shot", "article", best_k[m], TEST_TAG, main_scores[m]) for m in MODELS]
summary += [summary_row(m, label, setup, k, PROBE_TAG, probe_scores[(m, label)])
            for m in MODELS for label, setup, k in (("zero-shot", "article", 0), ("closed-book", "closedbook", best_k[m]))]
summary = pd.DataFrame(summary)
summary.to_csv(OUTPUT_DIR / "llm-test-runs.csv", index=False)
(OUTPUT_DIR / "llm-test-runs.md").write_text(to_markdown(summary))
summary.style.format(precision=4, na_rep="")

,model,setup,k,posts,BLEU,BLEU unsnapped,phrase,passage,multi,METEOR,BERTScore,type bal. acc.,exact / fuzzy / kept,unparsed,minutes,cost $
0,llama3.1-8b,few-shot,0,1000,0.1978,0.1938,0.2701,0.1597,0.1103,0.2286,0.8646,0.5451,93% / 6% / 1%,17,79.0000,
1,gpt-oss-20b,few-shot,0,1000,0.3541,0.3457,0.5851,0.1893,0.1743,0.4046,0.9052,0.4879,86% / 9% / 6%,0,43.2000,
2,claude-sonnet-5.5,few-shot,6,1000,0.4871,0.4833,0.5911,0.4225,0.3837,0.5056,0.9291,0.7348,98% / 2% / 0%,0,122.4000,1.6393
3,claude-haiku-5.5,few-shot,3,1000,0.3655,0.3593,0.4407,0.3280,0.2694,0.4020,0.9097,0.6525,95% / 4% / 0%,0,122.4000,0.0872
4,gpt-6.1-sol,few-shot,6,1000,0.5147,0.5143,0.6436,0.4428,0.3678,0.5079,0.9355,0.7507,100% / 0% / 0%,0,126.7000,5.2119
5,gpt-6-luna,few-shot,3,1000,0.4620,0.4645,0.6241,0.3517,0.3235,0.4826,0.9264,0.6649,96% / 1% / 3%,0,122.3000,0.0976
6,llama3.1-8b,zero-shot,0,300,0.1842,,0.2639,0.1386,0.0956,0.2139,0.8631,0.5396,92% / 6% / 2%,5,24.0000,
7,llama3.1-8b,closed-book,0,300,0.0198,,0.0467,0.0000,0.0000,0.0832,0.8600,0.6243,,0,3.8000,
8,gpt-oss-20b,zero-shot,0,300,0.3415,,0.5530,0.1965,0.1621,0.4154,0.9006,0.5215,87% / 6% / 7%,0,13.3000,
9,gpt-oss-20b,closed-book,0,300,0.0295,,0.0676,0.0011,0.0026,0.0895,0.8647,0.4840,,0,12.9000,


## A few test answers

Six test posts with the gold spoiler and every model's answer (snapped), to sanity-check the numbers.

In [8]:
preds_by_model = {m: {p["uuid"]: p for p in load_jsonl(PRED_DIR / f"{llm.run_name(m, 'article', best_k[m], TEST_TAG)}.jsonl")}
                  for m in MODELS}
show = llm.stratified_sample(test_main, 6, seed=3)
pd.DataFrame([{"post": post_text(r), "gold type": spoiler_type(r), "gold": as_text(r["spoiler"]),
               **{m: f"[{preds_by_model[m][r['uuid']]['spoilerType']}] {as_text(preds_by_model[m][r['uuid']]['spoiler'])}"
                  for m in MODELS}} for r in show])

,post,gold type,gold,llama3.1-8b,gpt-oss-20b,claude-sonnet-5.5,claude-haiku-5.5,gpt-6.1-sol,gpt-6-luna
0,A Major Musician Keeps Trying To Be On The Walking Dead,phrase,Chris Daughtry,[passage] Honestly I’ve been lobbying for The Walking Dead. I’ve been auditioning for ...,[phrase] Chris Daughtry has not yet landed a role on The Walking Dead.,[phrase] Chris Daughtry,[phrase] Chris Daughtry,[phrase] Chris Daughtry,[phrase] Chris Daughtry
1,Astronomers tally HOW MANY alien worlds beyond our solar system?,phrase,986,"[phrase] 1,000",[phrase] 986,[phrase] 986,[phrase] 986,[phrase] 986,[phrase] 986
2,"The best TV show of every year since 2000, according to critics",multi,"2000: ""The Sopranos"" (Season 2) 2001: ""The Office: UK"" (Season 1) 2002: ""The Office: U...",[multi] The Sopranos The Office: UK The Office: UK The Office: UK The Wire Deadwood Th...,"[multi] 2000: ""The Sopranos"" (Season 2) 2001: ""The Office: UK"" (Season 1) 2002: ""The O...","[multi] 2000: ""The Sopranos"" (Season 2) 2001: ""The Office: UK"" (Season 1) 2004: ""The W...","[multi] ""The Sopranos"" (Season 2) ""The Office: UK"" (Season 1) ""The Office: UK"" (Season...","[multi] 2000: ""The Sopranos"" (Season 2) 2001: ""The Office: UK"" (Season 1) 2002: ""The O...","[multi] 2000: ""The Sopranos"" (Season 2) Critic score: 97/100 User score: 9.3/10 What c..."
3,Pope Francis racks up another surprising magazine cover,phrase,Rolling Stone,[phrase] Cool Pope Francis,[phrase] first pope to ever be featured,[phrase] Rolling Stone,[phrase] He is the first pope to ever be featured.,[phrase] Rolling Stone,[phrase] Rolling Stone
4,Could there be other Universes with different versions of ourselves out there?,passage,"When it comes to physics, this is one of the most exciting possibilities of all, but i...",[passage] there are not enough of them to give us alternate versions of ourselves; the...,[passage] there are not enough of them to give us alternate versions of ourselves; the...,[passage] number of Universes out there&#8202; &#8202;possibly with different laws tha...,"[passage] But theoretically, what we learn is even more tantalizing. the number of pos...","[passage] unless inflation has been occurring for a truly infinite amount of time, the...",[passage] number of Universes out there&#8202; &#8202;possibly with different laws tha...
5,"At First, Her Tattoo Reads ""I'm Fine."" Now Watch What Happens When She Stands Up...",passage,"from another angle, it can read ""save me.""","[passage] Miles’s tattoo is actually an ambigram. It reads ""I’m fine"" from most angles...",[phrase] save me,"[phrase] it can read ""save me.""","[passage] It reads ""I’m fine"" from most angles, but if you look at from another angle,...",[phrase] save me,[phrase] save me
